# Member 3 on Kaggle: train the lite students (GPU)

Settings: **GPU T4 / P100**, **Internet on**. The notebook downloads `urbanheat_v1.h5` (2.2 GB) and the teacher from
Member 2's shared Drive folder. Every run trains a lite student on inputs with NDVI / NDBI / MNDWI clipped to [-1, 1];
`alpha < 1` distils the Member 2 teacher (`fusion_cnn` seed 0). Download `member3_runs.zip` when it finishes.

In [ ]:
# 1. code: the member3/efficient branch (edit REPO if the code comes as a Kaggle dataset instead)
REPO = "https://github.com/sebin-droid/Hyperspectral-and-Thermal.git"
!git clone -q -b member3/efficient {REPO} /kaggle/working/repo
!pip install -q -e /kaggle/working/repo
import sys; sys.path.insert(0, "/kaggle/working/repo/src")   # editable install is only seen after a restart

In [ ]:
# 2. data: Member 2's shared Drive folder (dataset + teacher run only)
from pathlib import Path
from urbanheat.data import drive
DRIVE = "1s04pea82m3Sg0JlElq-a1T1-Y_Oy3tkb"
DATA = Path("/kaggle/working/data")
files = [e for e in drive.walk(DRIVE) if not e.is_folder
         and (e.path == "urbanheat_v1.h5" or e.path.startswith("runs/cnn_robust_fusion_cnn_s0/"))]
for e in files:
    print(e.path, drive.download(e.id, DATA / e.path))
H5, TEACHER = DATA / "urbanheat_v1.h5", DATA / "runs/cnn_robust_fusion_cnn_s0"
RUNS = Path("/kaggle/working/runs"); RUNS.mkdir(exist_ok=True)
print(H5.stat().st_size / 1e9, "GB")

In [ ]:
# 3. train: same schedule as Member 2's robustness runs, so the comparison is fair
import json, torch
from urbanheat.efficient import distill
TRAIN = dict(epochs=60, batch_size=32, lr=2e-3, samples_per_epoch=4000, patience=10)
EXPERIMENTS = [  # (name, variant, width, thermal_width, alpha)
    ("m3_lite32_kd", "fusion_cnn", 32, 16, 0.5), ("m3_lite32_ce", "fusion_cnn", 32, 16, 1.0),
    ("m3_lite16_kd", "fusion_cnn", 16, 8, 0.5), ("m3_lite16_ce", "fusion_cnn", 16, 8, 1.0),
    ("m3_nohs32_kd", "fusion_nohs_cnn", 32, 16, 0.5),
]
results = {}
for name, variant, w, tw, alpha in EXPERIMENTS:
    if (RUNS / name / "metrics.json").exists():          # resume after a disconnect
        results[name] = json.loads((RUNS / name / "metrics.json").read_text()); continue
    torch.manual_seed(0)
    m, _ = distill.run(H5, RUNS / name, TEACHER, alpha=alpha, variant=variant, width=w, thermal_width=tw, device="cuda", **TRAIN)
    results[name] = m
    print(f"{name}: test R2 {m['test']['r2']:.3f}  MAE {m['test']['mae']:.2f}  params {m['params']:,}  "
          f"MFLOPs {m['flops'] / 1e6:.1f}  epochs {m['epochs_run']}")

In [ ]:
# 4. table
import pandas as pd
pd.DataFrame({n: dict(test_r2=r["test"]["r2"], test_mae=r["test"]["mae"], test_rmse=r["test"]["rmse"],
                      val_r2=r["val"]["r2"], params=r["params"], mflops=r["flops"] / 1e6,
                      gpu_ms_per_patch=r["latency_ms"], peak_mem_mb=r["peak_mem_mb"]) for n, r in results.items()}).T.round(3)

In [ ]:
# 5. pack the outputs for download
!cd /kaggle/working && zip -qr member3_runs.zip runs && ls -lh member3_runs.zip